# Obstacles across the seven resolutions

This notebook shows, at every resolution, one validation scene where the obstacles change the optimal
route. For each resolution it takes the first `candidates` validation preview scenes and keeps the one
where blocking the obstacles moves the route furthest, measured by the 95th-percentile Hausdorff
distance. The selection rule is fixed in advance; it picks clear illustrations, not typical scenes.
The typical effect of the obstacles is measured in the `obstacles` and `metrics` notebooks.

**Run it yourself.** From a checkout of the [code repository](https://github.com/ThalesGroup/passage), start Jupyter with `uv run --frozen jupyter lab`, open this notebook and point `dataset_path` to a local copy of [the dataset](https://huggingface.co/datasets/thalesgroup/passage). With `dataset_path = None`, rows are read from the Hub at the pinned `dataset_revision` instead. It decodes up to 16 preview scenes per resolution; a few minutes in total.

In [ ]:
resolutions = [64, 128, 256, 512, 1024, 2048, 4096]
split = "validation"
objective = "elevation"  # elevation, slope_uphill or energy
candidates = 16  # preview scenes examined per resolution
output_dir = "notebook-artifacts/notebooks/paper/obstacles_showcase"
dataset_path = "dataset"  # local copy of the dataset; None reads the Hub at dataset_revision
dataset_repo = "thalesgroup/passage"
dataset_revision = "2259dd94e2de28ab4a56c270247a7e347ec8f133"  # verified PASSAGE v1.0 upload
project_path = None  # checkout root; None searches upwards from the working directory

In [ ]:
from pathlib import Path
import json  # noqa: F401
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np  # noqa: F401
import pandas as pd
from IPython.display import Markdown, display

# Locate the code checkout (pyproject.toml next to src/passage) without changing directory.
project_root = Path(project_path).resolve() if project_path else next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "src/passage").is_dir()), Path.cwd())
for extra in (project_root / "src", project_root / "scripts"):
    if extra.is_dir() and str(extra) not in sys.path:
        sys.path.insert(0, str(extra))

from passage.notebook_reports import make_source, resolve_path, save_figure, write_report  # noqa: E402, F401

source = make_source(dataset_path, dataset_repo=dataset_repo, dataset_revision=dataset_revision,
                     project_root=project_root)
output_path = resolve_path(output_dir, project_root)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 30)


def relative(path):
    """Show saved files relative to the checkout, never as machine paths."""
    path = Path(path)
    try:
        return path.resolve().relative_to(project_root.resolve()).as_posix()
    except ValueError:
        return path.name


def show(frame, caption=None, floatfmt="{:,.3f}"):
    """Display a table with readable numbers and an optional caption."""
    if caption:
        display(Markdown(f"*{caption}*"))
    floats = list(frame.select_dtypes("float").columns)
    ints = list(frame.select_dtypes("integer").columns)
    others = [c for c in frame.columns if c not in floats and c not in ints]
    styler = frame.style.hide(axis="index")
    for columns, fmt in ((floats, floatfmt), (ints, "{:,}"), (others, None)):
        if columns:
            styler = styler.format(fmt, subset=columns, na_rep="—")
    display(styler)


# One chart style for every PASSAGE notebook: recessive axes, light grid, left-aligned titles.
INK, MUTED, GRID = "#243537", "#5c6765", "#e3e6e3"
OBJECTIVE_COLORS = {"elevation": "#1767ac", "slope_uphill": "#d56418", "energy": "#a32b8e"}
OBJECTIVE_LABELS = {"elevation": "Elevation", "slope_uphill": "Slope (uphill)", "energy": "Energy"}
SPLIT_COLORS = {"train": "#1f8a70", "calibration": "#b07d1a", "validation": "#3a62b5", "test": "#c0392b"}
REGIME_COLORS = {"free": "#238b45", "obstacles": "#1767ac"}
plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 9, "axes.edgecolor": "#8a9290", "axes.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "axes.grid.axis": "y",
    "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True, "axes.titlelocation": "left",
    "axes.titlesize": 10.5, "axes.titleweight": "semibold", "axes.titlecolor": INK, "axes.labelcolor": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "legend.frameon": False, "lines.linewidth": 1.6,
    "figure.dpi": 110, "savefig.dpi": 140, "pdf.fonttype": 42, "ps.fonttype": 42,
})


def figure_title(fig, title, subtitle=None):
    """Left-aligned title and subtitle in a band reserved at the top of the figure."""
    height = fig.get_figheight()
    fig.suptitle(title, x=0.01, y=1 - 0.1 / height, ha="left", va="top", fontsize=12, fontweight="semibold", color=INK)
    if subtitle:
        fig.text(0.01, 1 - 0.4 / height, subtitle, ha="left", va="top", fontsize=8.5, color=MUTED)
    fig.subplots_adjust(top=min(fig.subplotpars.top, 1 - 0.95 / height))


# The shared trajectory style of the paper: one terrain scale, white start circle and goal star,
# green solid routes on free terrain and blue short dashes when obstacles are blocked.
from trajectory_style import render_cost_comparison, draw_comparison, comparison_handles  # noqa: E402, F401
from passage.plotting import decode_scene, render_scene, terrain_colormap, OBSTACLE_COLOR  # noqa: E402, F401


def first_scene(resolution, split, min_share=0.25, min_coverage=0.05):
    """Selection rule fixed in advance: the first scene of the split whose six stored routes all span at least
    `min_share` of the grid side and whose obstacles cover at least `min_coverage` of the cells."""
    for candidate in source.iter_rows(resolution, split, columns=["id", "metadata"], samples=False):
        lengths = [variant["length"] for paths in candidate["metadata"]["paths"].values()
                   for variant in (paths["free"], paths["obstacles"])]
        coverage = candidate["metadata"]["obstacles"]["ratio_actual"]
        if min(lengths) >= min_share * resolution and coverage >= min_coverage:
            return candidate["id"]
    raise ValueError("No scene of the split meets the selection rule")

## 1. Picking one scene per resolution

In [ ]:
from passage.notebook_reports import route_metrics_report

SOLVER = {"elevation": "astar_elevation", "slope_uphill": "astar_slope", "energy": "astar_energy"}[objective]
picked, rows = {}, []
for resolution in resolutions:
    best = None
    for preview in source.iter_rows(resolution, split, columns=None, samples=True, limit=candidates):
        result = route_metrics_report(preview, resolution, split)
        pair = next(c for c in result["comparisons"]
                    if {c["route"], c["reference_route"]} == {f"path_{SOLVER}_free", f"path_{SOLVER}_obstacles"})
        if pair["hausdorff_p95_pixels"] is None:
            continue
        if best is None or pair["hausdorff_p95_pixels"] > best[1]["hausdorff_p95_pixels"]:
            best = (preview, pair, result)
    preview, pair, result = best
    picked[resolution] = preview
    cost = {r["task_variant"]: r["stored_cost"] for r in result["routes"] if r["solver"] == SOLVER}
    cells = {r["task_variant"]: r["vertices"] for r in result["routes"] if r["solver"] == SOLVER}
    rows.append({"Resolution": f"{resolution}×{resolution}", "Scene": preview["id"],
                 "Obstacle coverage (%)": 100 * preview["metadata"]["obstacles"]["ratio_actual"],
                 "Free route (cells)": cells["free"], "Obstacle route (cells)": cells["obstacles"],
                 "Detour (%)": 100 * (cost["obstacles"] / cost["free"] - 1),
                 "H95 (% of side)": 100 * pair["hausdorff_p95_pixels"] / resolution})
chosen = pd.DataFrame(rows)
show(chosen, f"Table 1. Scene shown per resolution ({OBJECTIVE_LABELS[objective]} objective).", "{:,.2f}")

## 2. The showcase

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(12.5, 7.2))
fig.subplots_adjust(top=0.84, bottom=0.04, hspace=0.32, wspace=0.08)
for ax, (resolution, preview), info in zip(axes.flat, picked.items(), rows):
    elevation, obstacles, start, goal = decode_scene(preview)
    draw_comparison(elevation, obstacles, tuple(start), tuple(goal), preview[f"path_{SOLVER}_free"],
                    preview[f"path_{SOLVER}_obstacles"], ax=ax, labels=("Free terrain", "With obstacles"))
    ax.set_title(f"{resolution}×{resolution}\n{info['Obstacle coverage (%)']:.0f} % blocked · "
                 f"detour +{info['Detour (%)']:.1f} %", fontsize=9, pad=12)
axes.flat[-1].axis("off")
axes.flat[-1].legend(handles=comparison_handles(("Free terrain", "With obstacles")), loc="center", fontsize=9)
figure_title(fig, "One scene per resolution where the obstacles move the route",
             f"{OBJECTIVE_LABELS[objective]} objective · one {split} scene per resolution · titles give coverage and detour")
figures = save_figure(fig, output_path, f"obstacles_showcase-{objective}")
plt.show()

*Figure 1. For each resolution, the free-terrain optimum (green, solid) and the obstacle-aware optimum
(blue, dashed) of one scene. Panel titles give the obstacle coverage and the extra cost of the
obstacle-aware route. Grids larger than 512 cells per side are drawn on a 512-cell preview.*

## Summary

In [ ]:
display(Markdown("\n".join([
    f"- In the selected scenes, the obstacles move the {OBJECTIVE_LABELS[objective].lower()} route by "
    f"{chosen['H95 (% of side)'].min():.1f} to {chosen['H95 (% of side)'].max():.1f} % of the grid side, "
    f"for an extra cost of {chosen['Detour (%)'].min():.2f} to {chosen['Detour (%)'].max():.2f} %.",
    "- A large move with a small extra cost means that an almost equally good corridor existed elsewhere: "
    "planners must find it, not only repair the free-terrain route locally.",
])))
report = {"objective": objective, "candidates": candidates, "chosen": chosen.to_dict("records"),
          "figures": [relative(f) for f in figures]}

In [ ]:
report_file = write_report(report, output_path, f"obstacles_showcase-{objective}.json")
display(Markdown(f"The numbers of this run are saved in `{relative(report_file)}`, next to the figures."))

---
*MIT License · Copyright (c) 2026 THALES. The full licence text is in the `LICENSE` file of the repository.*